In [ ]:
# ============================================================
# CELL 1 — Data Preprocessing: Build PKL (Window = 99 min)
# Training houses: 2, 3, 5, 9 | Validation: House 20
# ON threshold: 30W | Stride train: 2 | Stride val: 1
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


import os
path = '/content/drive/MyDrive/new obada/data/labeled/fridge.csv'
print(" ✅" if os.path.exists(path) else "  ❌")
import os, pickle, gc
import numpy as np
import pandas as pd

BASE         = '/content/drive/MyDrive/new obada/'
LABELED_PATH = BASE + 'data/labeled/'
PKL_OUT      = BASE + 'data/preprocessed/'

WINDOW_SIZE  = 99
TRAIN_HOUSES = [2, 3, 5, 9]
VAL_HOUSE    = 20
STRIDE_TRAIN = 2
STRIDE_VAL   = 1
RANDOM_SEED  = 42
ON_THRESHOLD = 30

def extract_windows(agg_arr, label_arr, stride):
    half = WINDOW_SIZE // 2
    X, y = [], []
    n = len(agg_arr)
    for start in range(0, n - WINDOW_SIZE + 1, stride):
        end    = start + WINDOW_SIZE
        centre = start + half
        if np.any(np.isnan(agg_arr[start:end])):
            continue
        X.append(agg_arr[start:end])
        y.append(label_arr[centre])
    if not X:
        return np.array([]), np.array([])
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int8))

print("Loading fridge.csv ...")
df = pd.read_csv(LABELED_PATH + 'fridge.csv', parse_dates=['timestamp'])
df['fridge_label'] = (df['fridge_watts'] >= ON_THRESHOLD).astype(int)

# ── Train ─────────────────────────────────────────────────
train_out = PKL_OUT + 'fridge_train_v4.pkl'
if os.path.exists(train_out):
    print("⏭ train v4 exists")
else:
    all_X, all_y = [], []
    for hid in TRAIN_HOUSES:
        hdf  = df[df['house_id'] == hid].reset_index(drop=True)
        agg  = hdf['aggregate_watts'].values.astype(np.float32)
        mn, mx = np.nanmin(agg), np.nanmax(agg)
        if mx - mn < 1e-6: continue
        agg_norm = (agg - mn) / (mx - mn)
        labels   = hdf['fridge_label'].values.astype(np.int8)
        X, y = extract_windows(agg_norm, labels, STRIDE_TRAIN)
        if len(X) == 0: continue
        print(f"  House {hid}: {len(X):,} windows | "
              f"ON={y.sum():,} ({100*y.mean():.1f}%)")
        all_X.append(X); all_y.append(y)
        del X, y, agg_norm; gc.collect()

    X_all = np.concatenate(all_X)
    y_all = np.concatenate(all_y)
    del all_X, all_y; gc.collect()

    # Balance 50/50
    rng    = np.random.default_rng(RANDOM_SEED)
    idx_on  = np.where(y_all == 1)[0]
    idx_off = np.where(y_all == 0)[0]
    n_each  = min(len(idx_on), len(idx_off))
    keep = np.sort(np.concatenate([
        rng.choice(idx_on,  n_each, replace=False),
        rng.choice(idx_off, n_each, replace=False),
    ]))
    X_bal, y_bal = X_all[keep], y_all[keep]
    del X_all, y_all; gc.collect()

    with open(train_out, 'wb') as f:
        pickle.dump({'X': X_bal, 'y_label': y_bal}, f, protocol=4)
    size = os.path.getsize(train_out) / 1e6
    print(f"✅ Train: shape={X_bal.shape} | "
          f"ON={y_bal.sum():,} (50%) | {size:.0f} MB")
    del X_bal, y_bal; gc.collect()

# ── Val ───────────────────────────────────────────────────
val_out = PKL_OUT + 'fridge_val_v4.pkl'
if os.path.exists(val_out):
    print("⏭ val v4 exists")
else:
    hdf  = df[df['house_id'] == VAL_HOUSE].reset_index(drop=True)
    agg  = hdf['aggregate_watts'].values.astype(np.float32)
    mn, mx = np.nanmin(agg), np.nanmax(agg)
    agg_norm = (agg - mn) / (mx - mn)
    labels   = hdf['fridge_label'].values.astype(np.int8)
    X_val, y_val = extract_windows(agg_norm, labels, STRIDE_VAL)
    print(f"  House 20: {len(X_val):,} windows | "
          f"ON={y_val.sum():,} ({100*y_val.mean():.1f}%)")
    with open(val_out, 'wb') as f:
        pickle.dump({'X': X_val, 'y_label': y_val}, f, protocol=4)
    size = os.path.getsize(val_out) / 1e6
    print(f"✅ Val: shape={X_val.shape} | {size:.0f} MB")
    del X_val, y_val; gc.collect()

# ── Sanity ────────────────────────────────────────────────
print("\n── Sanity Check ──")
for name in ['fridge_train_v4.pkl', 'fridge_val_v4.pkl']:
    with open(PKL_OUT + name, 'rb') as f:
        d = pickle.load(f)
    print(f"  {name}: shape={d['X'].shape} | "
          f"ON={d['y_label'].sum():,} ({100*d['y_label'].mean():.1f}%)")
    del d; gc.collect()

print("\n✅   V4")

Mounted at /content/drive
موجود ✅
Loading fridge.csv ...
  House 2: 337,565 windows | ON=135,849 (40.2%)
  House 3: 393,186 windows | ON=204,448 (52.0%)
  House 5: 427,890 windows | ON=220,939 (51.6%)
  House 9: 346,803 windows | ON=173,981 (50.2%)
✅ Train: shape=(1470434, 99) | ON=735,217 (50%) | 584 MB
  House 20: 623,919 windows | ON=261,584 (41.9%)
✅ Val: shape=(623919, 99) | 248 MB

── Sanity Check ──
  fridge_train_v4.pkl: shape=(1470434, 99) | ON=735,217 (50.0%)
  fridge_val_v4.pkl: shape=(623919, 99) | ON=261,584 (41.9%)

✅ جاهز للتدريب V4


In [ ]:
# ============================================================
# CELL 2 — Model Training & Evaluation
# Architecture: CNN-BiLSTM (3 CNN layers + 2-layer BiLSTM)
# Window: 99 | Batch: 512 | Epochs: 30 | LR: 1e-3
# Result: F1=0.688 | Precision=0.545 | Recall=0.933 | Threshold=0.22
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")

# CELL 2 — Imports & Config
import os, pickle, gc, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

BASE      = '/content/drive/MyDrive/new obada/'
PKL_DIR   = BASE + 'data/preprocessed/'
MODEL_DIR = BASE + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

WINDOW_SIZE = 99
BATCH_SIZE  = 512
EPOCHS      = 30
LR          = 1e-3
WARMUP      = 3
RANDOM_SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# CELL 3 — Load Data
print("Loading fridge v4 PKLs...")
with open(PKL_DIR + 'fridge_train_v4.pkl', 'rb') as f:
    tr = pickle.load(f)
with open(PKL_DIR + 'fridge_val_v4.pkl', 'rb') as f:
    vl = pickle.load(f)

X_train = tr['X'].astype(np.float32)
y_train = tr['y_label'].astype(np.float32)
X_val   = vl['X'].astype(np.float32)
y_val   = vl['y_label'].astype(np.float32)

print(f"Train: {X_train.shape} | ON={y_train.mean():.3f}")
print(f"Val  : {X_val.shape}   | ON={y_val.mean():.4f}")

# Fixed stratified val sample
rng    = np.random.default_rng(RANDOM_SEED)
idx_on  = np.where(y_val == 1)[0]
idx_off = np.where(y_val == 0)[0]
n_each  = min(40_000, len(idx_on))
val_idx = np.sort(np.concatenate([
    rng.choice(idx_on,  n_each, replace=False),
    rng.choice(idx_off, n_each, replace=False),
]))
X_val_s = X_val[val_idx]
y_val_s = y_val[val_idx]
print(f"Val sample: {X_val_s.shape} | ON={y_val_s.mean():.3f}")

# CELL 4 — Dataset & Model
class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)  # (N, 1, 99)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


class FridgeCNNBiLSTM_V4(nn.Module):
 ──────────────────────────────────────────
    """
    def __init__(self, lstm_hidden=64, lstm_layers=2, dropout=0.3):
        super().__init__()

        # CNN encoder — smaller than kettle since window size is shorter (99 vs 599)
        self.cnn = nn.Sequential(
            nn.Conv1d(1,  32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )

        # NO sigmoid — BCEWithLogitsLoss
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        feat = self.cnn(x)                    # (B, 64, 99)
        feat = feat.permute(0, 2, 1)          # (B, 99, 64)
        out, _ = self.lstm(feat)              # (B, 99, 128)
        mid    = out[:, out.size(1)//2, :]   # centre point
        return self.head(mid).squeeze(1)


model = FridgeCNNBiLSTM_V4().to(device)
print(f"✅ Model params: {sum(p.numel() for p in model.parameters()):,}")

# CELL 5 — Training
train_loader = DataLoader(
    NILMDataset(X_train, y_train),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    NILMDataset(X_val_s, y_val_s),
    batch_size=2048, shuffle=False,
    num_workers=2, pin_memory=True,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_f1_on = 0.0
best_path  = MODEL_DIR + 'fridge_v4_best.pt'

print("=" * 75)
print(f"{'Ep':>3} {'Loss':>7} {'F1_ON':>7} {'F1_mac':>7} "
      f"{'Prec':>7} {'Rec':>7} {'Thr':>5} {'LR':>9}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    # TRAIN
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b = X_b.to(device, non_blocking=True)
        y_b = y_b.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # VALIDATION
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device, non_blocking=True)
            probs = torch.sigmoid(model(X_b)).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y_b.numpy().astype(int))

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_t, best_f1_ep = 0.5, 0.0
    for t in np.arange(0.20, 0.85, 0.05):
        preds = (all_probs >= t).astype(int)
        f1    = f1_score(all_labels, preds, pos_label=1,
                         average='binary', zero_division=0)
        if f1 > best_f1_ep:
            best_f1_ep, best_t = f1, t

    final_preds = (all_probs >= best_t).astype(int)
    f1_mac = f1_score(all_labels, final_preds, average='macro', zero_division=0)
    prec   = precision_score(all_labels, final_preds, zero_division=0)
    rec    = recall_score(all_labels, final_preds, zero_division=0)
    is_best = best_f1_ep > best_f1_on

    print(
        f"{epoch:3d} {train_loss/len(train_loader):7.4f} "
        f"{best_f1_ep:7.4f} {f1_mac:7.4f} "
        f"{prec:7.4f} {rec:7.4f} "
        f"{best_t:5.2f} {optimizer.param_groups[0]['lr']:9.6f}"
        + (" ✅ BEST" if is_best else "")
    )

    if is_best:
        best_f1_on = best_f1_ep
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_f1_on': best_f1_on,
            'threshold': best_t,
            'architecture': 'CNN-BiLSTM V4 Window=99',
            'window_size': 99,
            'appliance': 'fridge',
        }, best_path)

print("=" * 75)
print(f"\n🏆 Best F1_ON = {best_f1_on:.4f}")
print(f"💾 Saved → {best_path}")

# CELL 6 — Full Val + Threshold Sweep
full_loader = DataLoader(
    NILMDataset(X_val, y_val),
    batch_size=2048, shuffle=False,
    num_workers=2, pin_memory=True,
)

ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in full_loader:
        X_b = X_b.to(device, non_blocking=True)
        probs = torch.sigmoid(model(X_b)).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_b.numpy().astype(int))

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

results = []
print(f"\n{'Thr':>6} {'F1_ON':>10} {'Prec':>10} {'Recall':>10}")
print("=" * 45)
for t in np.arange(0.20, 0.95, 0.02):
    p = (all_probs >= t).astype(int)
    f1 = f1_score(all_labels, p, pos_label=1, average='binary', zero_division=0)
    pr = precision_score(all_labels, p, zero_division=0)
    rc = recall_score(all_labels, p, zero_division=0)
    results.append((t, f1, pr, rc))
    print(f"{t:6.2f} {f1:10.4f} {pr:10.4f} {rc:10.4f}")

best_r = max(results, key=lambda x: x[1])
on_p   = all_probs[all_labels == 1]
off_p  = all_probs[all_labels == 0]

print("\n" + "=" * 55)
print(f"BEST → Thr={best_r[0]:.2f}  F1={best_r[1]:.4f}  "
      f"Prec={best_r[2]:.4f}  Rec={best_r[3]:.4f}")
print(f"Separation = {on_p.mean() - off_p.mean():.4f}")
print(f"  ON  mean: {on_p.mean():.4f}")
print(f"  OFF mean: {off_p.mean():.4f}")
print("=" * 55)

ckpt['threshold_full_val'] = best_r[0]
ckpt['f1_full_val']        = best_r[1]
torch.save(ckpt, best_path)
print("✅ Checkpoint updated")

Mounted at /content/drive
✅ Drive mounted
✅ Device: cuda
Loading fridge v4 PKLs...
Train: (1470434, 99) | ON=0.500
Val  : (623919, 99)   | ON=0.4193
Val sample: (80000, 99) | ON=0.500
✅ Model params: 199,489
 Ep    Loss   F1_ON  F1_mac    Prec     Rec   Thr        LR
  1  0.4363  0.7294  0.6097  0.5908  0.9530  0.20  0.000667 ✅ BEST
  2  0.3449  0.7374  0.6397  0.6077  0.9375  0.20  0.001000 ✅ BEST
  3  0.2998  0.6978  0.5683  0.5670  0.9072  0.20  0.001000
  4  0.2677  0.7096  0.5824  0.5753  0.9256  0.25  0.000997
  5  0.2467  0.7482  0.6608  0.6210  0.9409  0.20  0.000987 ✅ BEST
  6  0.2313  0.6939  0.5975  0.5816  0.8599  0.20  0.000970
  7  0.2181  0.6967  0.6217  0.5978  0.8348  0.20  0.000947
  8  0.2067  0.6837  0.6202  0.5981  0.7980  0.20  0.000918
  9  0.1971  0.7390  0.6706  0.6306  0.8923  0.20  0.000883
 10  0.1887  0.6702  0.6198  0.6000  0.7590  0.20  0.000843
 11  0.1815  0.7310  0.6602  0.6235  0.8833  0.20  0.000799
 12  0.1739  0.7369  0.6558  0.6189  0.9106  0.20  